# Task 3.5

This notebook answers Task 3.5. It solves the full problem: minimize the cost of brewing 5000 litres of amber beer while requiring that the mean qualities of the malt sources, hops, and yeast are each at least 3.

In [15]:
from brew_utils import *
from scipy.optimize import linprog
import numpy as np
import pandas as pd

## 3.5 Full Problem Setup

This task extends the amber-beer model from Task 3.3 by adding minimum quality constraints. The target is still 5000 litres of amber beer, but now the average quality of malt sources, hops, and yeast must each be at least 3.

In [16]:
barley_cost = barley["cost"].to_numpy(dtype=float)
barley_ebc = barley["EBC"].to_numpy(dtype=float)
barley_q = barley["quality"].to_numpy(dtype=float)

malt_cost = malts["cost"].to_numpy(dtype=float)
malt_ebc = malts["EBC"].to_numpy(dtype=float)
malt_q = malts["quality"].to_numpy(dtype=float)

extract_cost = maltextracts["cost"].to_numpy(dtype=float)
extract_ebc = maltextracts["EBC"].to_numpy(dtype=float)
extract_q = maltextracts["quality"].to_numpy(dtype=float)

hop_cost = hops["cost"].to_numpy(dtype=float)
hop_q = hops["quality"].to_numpy(dtype=float)

yeast_cost = yeasts["cost"].to_numpy(dtype=float)
yeast_q = yeasts["quality"].to_numpy(dtype=float)

F_malt = float(maltprocess.loc[0, "fixedcost"])
v_malt = float(maltprocess.loc[0, "variablecost"])
F_mash = float(mashingprocess.loc[0, "fixedcost"])
v_mash = float(mashingprocess.loc[0, "variablecost"])

nB = len(barley)
nM = len(malts)
nE = len(maltextracts)
nH = len(hops)
nY = len(yeasts)

iB0 = 0
iM0 = iB0 + nB
iE0 = iM0 + nM
iH0 = iE0 + nE
iY0 = iH0 + nH
iZm = iY0 + nY
iZs = iZm + 1
nvars = iZs + 1

def slice_B():
    return slice(iB0, iB0 + nB)

def slice_M():
    return slice(iM0, iM0 + nM)

def slice_E():
    return slice(iE0, iE0 + nE)

def slice_H():
    return slice(iH0, iH0 + nH)

def slice_Y():
    return slice(iY0, iY0 + nY)

## 3.5 Model with Quality Constraints

This block builds the full MILP. Compared with Task 3.3, it adds three lower-bound quality constraints:
- average malt-source quality at least `Q_malt`
- average hop quality at least `Q_hop`
- average yeast quality at least `Q_yeast`

The objective remains to minimize total cost.

In [17]:
def build_lp_model(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast, z_fix=None):
    m_tot = malt_extract_eq_needed_for_beer(V_beer)
    H_tot = hops_needed_for_beer(V_beer)
    Y_tot = yeast_needed_for_beer(V_beer)
    gamma = (SG - 1.0) / 0.0344

    M_B = m_tot / BARLEY_TO_MALTEXTRACT_EQ
    M_M = m_tot / MALT_TO_MALTEXTRACT_EQ

    c = np.zeros(nvars)
    c[slice_B()] = barley_cost + v_malt + v_mash * BARLEY_TO_MALT
    c[slice_M()] = malt_cost + v_mash
    c[slice_E()] = extract_cost
    c[slice_H()] = hop_cost
    c[slice_Y()] = yeast_cost
    c[iZm] = F_malt
    c[iZs] = F_mash

    A_eq = []
    b_eq = []

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALTEXTRACT_EQ
    row[slice_M()] = MALT_TO_MALTEXTRACT_EQ
    row[slice_E()] = 1.0
    A_eq.append(row)
    b_eq.append(m_tot)

    row = np.zeros(nvars)
    row[slice_H()] = 1.0
    A_eq.append(row)
    b_eq.append(H_tot)

    row = np.zeros(nvars)
    row[slice_Y()] = 1.0
    A_eq.append(row)
    b_eq.append(Y_tot)

    A_ub = []
    b_ub = []

    if np.isfinite(EBC_max):
        row = np.zeros(nvars)
        row[slice_B()] = gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
        row[slice_M()] = gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
        row[slice_E()] = gamma * extract_ebc
        A_ub.append(row)
        b_ub.append(EBC_max * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = -gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
    row[slice_M()] = -gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
    row[slice_E()] = -gamma * extract_ebc
    A_ub.append(row)
    b_ub.append(-EBC_min * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = -BARLEY_TO_MALTEXTRACT_EQ * barley_q
    row[slice_M()] = -MALT_TO_MALTEXTRACT_EQ * malt_q
    row[slice_E()] = -extract_q
    A_ub.append(row)
    b_ub.append(-Q_malt * m_tot)

    row = np.zeros(nvars)
    row[slice_H()] = -hop_q
    A_ub.append(row)
    b_ub.append(-Q_hop * H_tot)

    row = np.zeros(nvars)
    row[slice_Y()] = -yeast_q
    A_ub.append(row)
    b_ub.append(-Q_yeast * Y_tot)

    row = np.zeros(nvars)
    row[slice_B()] = 1.0
    row[iZm] = -M_B
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALT
    row[slice_M()] = 1.0
    row[iZs] = -M_M
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[iZm] = 1.0
    row[iZs] = -1.0
    A_ub.append(row)
    b_ub.append(0.0)

    bounds = [(0, None)] * nvars
    if z_fix is None:
        bounds[iZm] = (0, 1)
        bounds[iZs] = (0, 1)
    else:
        bounds[iZm] = (z_fix[0], z_fix[0])
        bounds[iZs] = (z_fix[1], z_fix[1])

    return {
        "c": c,
        "A_eq": np.array(A_eq, dtype=float),
        "b_eq": np.array(b_eq, dtype=float),
        "A_ub": np.array(A_ub, dtype=float),
        "b_ub": np.array(b_ub, dtype=float),
        "bounds": bounds,
        "m_tot": m_tot,
        "H_tot": H_tot,
        "Y_tot": Y_tot,
    }

## 3.5 Solve the Full Problem

The original MILP is solved by brute force over the valid process choices:
- `(0, 0)` only malt extract
- `(0, 1)` mashing only
- `(1, 1)` malting and mashing

For each case, the process variables are fixed and the remaining problem is solved as a linear program.

In [18]:
def solve_with_fixed_processes(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast, z_malt, z_mash):
    model = build_lp_model(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast, z_fix=(z_malt, z_mash))
    res = linprog(
        c=model["c"],
        A_ub=model["A_ub"],
        b_ub=model["b_ub"],
        A_eq=model["A_eq"],
        b_eq=model["b_eq"],
        bounds=model["bounds"],
        method="highs",
    )
    return model, res

def solve_bruteforce(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast):
    candidates = [(0, 0), (0, 1), (1, 1)]
    best = None
    all_results = []

    for z_fix in candidates:
        model, res = solve_with_fixed_processes(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast, *z_fix)
        info = {
            "z_fix": z_fix,
            "success": res.success,
            "cost": res.fun if res.success else np.nan,
            "result": res,
            "model": model,
        }
        all_results.append(info)

        if res.success and (best is None or res.fun < best["cost"]):
            best = info

    return best, all_results

## 3.5 Reporting Helpers

These functions summarize the final beer properties and the non-zero decision variables so the result can be compared directly with Task 3.3.

In [19]:
def solution_to_dataframe(x, tol=1e-9):
    names = (
        list(barley["name"])
        + list(malts["name"])
        + list(maltextracts["name"])
        + list(hops["name"])
        + list(yeasts["name"])
        + ["z_malt", "z_mash"]
    )
    df = pd.DataFrame({"variable": names, "value": x})
    return df[df["value"].abs() > tol].reset_index(drop=True)

def summarize_solution(model, res):
    x = res.x
    b = x[slice_B()]
    m = x[slice_M()]
    e = x[slice_E()]
    h = x[slice_H()]
    y = x[slice_Y()]

    m_tot = model["m_tot"]
    H_tot = model["H_tot"]
    Y_tot = model["Y_tot"]
    gamma = (SG - 1.0) / 0.0344

    ebc_value = gamma * (
        BARLEY_TO_MALTEXTRACT_EQ * np.sum(barley_ebc * b)
        + MALT_TO_MALTEXTRACT_EQ * np.sum(malt_ebc * m)
        + np.sum(extract_ebc * e)
    ) / m_tot if m_tot > 0 else 0.0

    q_malt_value = (
        BARLEY_TO_MALTEXTRACT_EQ * np.sum(barley_q * b)
        + MALT_TO_MALTEXTRACT_EQ * np.sum(malt_q * m)
        + np.sum(extract_q * e)
    ) / m_tot if m_tot > 0 else 0.0

    q_hop_value = np.sum(hop_q * h) / H_tot if H_tot > 0 else 0.0
    q_yeast_value = np.sum(yeast_q * y) / Y_tot if Y_tot > 0 else 0.0

    return {
        "cost": res.fun,
        "EBC": ebc_value,
        "Q_malt": q_malt_value,
        "Q_hop": q_hop_value,
        "Q_yeast": q_yeast_value,
        "z_malt": x[iZm],
        "z_mash": x[iZs],
    }

def print_solution(title, model, res):
    print(f"\n=== {title} ===")
    print("Success:", res.success)
    print("Message:", res.message)
    if not res.success:
        return
    summary = summarize_solution(model, res)
    for k, v in summary.items():
        print(f"{k}: {v}")
    print("\nNon-zero variables:")
    print(solution_to_dataframe(res.x).to_string(index=False))

## 3.5 Run the Full Problem

This block runs the full problem for 5000 litres of amber beer with minimum quality 3 in each ingredient group.

In [20]:
V_beer = 5000
EBC_min = 20
EBC_max = 29
Q_malt = 3
Q_hop = 3
Q_yeast = 3

In [21]:
best_solution, all_results = solve_bruteforce(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast)

print("Brute-force cases:")
print(pd.DataFrame([
    {"z_fix": r["z_fix"], "success": r["success"], "cost": r["cost"]}
    for r in all_results
]).to_string(index=False))

if best_solution is not None:
    print_solution("3.5 Full problem solution", best_solution["model"], best_solution["result"])

Brute-force cases:
 z_fix  success        cost
(0, 0)     True 7434.583344
(0, 1)     True 4891.923363
(1, 1)     True 5588.556474

=== 3.5 Full problem solution ===
Success: True
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
cost: 4891.923363286267
EBC: 20.000000000000007
Q_malt: 2.999999999999999
Q_hop: 3.0
Q_yeast: 3.0
z_malt: 0.0
z_mash: 1.0

Non-zero variables:
variable      value
  malt03 901.456996
  malt06  17.496791
  malt12  34.993582
   hop07   3.421053
   hop10   3.421053
 yeast03   3.750000
  z_mash   1.000000


## 3.5 Compare with Task 3.3

Compared with Task 3.3, this solution includes minimum quality constraints. That can force the model to choose more expensive ingredients, or a different process configuration, in order to keep the average malt-source, hop, and yeast qualities at or above 3.

<!-- REPORT_TABLE_OUTPUTS_SYNC -->
## Report table outputs

The following tables correspond to the tables used in the LaTeX report for Task 3.5.

In [22]:
# REPORT_TABLE_OUTPUTS_SYNC
print("Table 8.1: Task 5 Problem Setup")
task35_setup_table = pd.DataFrame([
    {"Parameter": "Beer volume", "Value": "5000 L"},
    {"Parameter": "Colour", "Value": "amber"},
    {"Parameter": "EBC bounds", "Value": "20 to 29"},
    {"Parameter": "Malt-source quality", "Value": "at least 3"},
    {"Parameter": "Hop quality", "Value": "at least 3"},
    {"Parameter": "Yeast quality", "Value": "at least 3"},
])
display(task35_setup_table)

summary = summarize_solution(best_solution["model"], best_solution["result"])
print("Table 8.2: Task 5 Optimal Solution Summary")
task35_solution_table = pd.DataFrame([
    {"Item": "Total cost", "Result": f"{summary['cost']:.3f}"},
    {"Item": "Process choice", "Result": f"({int(round(summary['z_malt']))},{int(round(summary['z_mash']))})"},
    {"Item": "Achieved EBC", "Result": f"{summary['EBC']:.3f}"},
    {"Item": "Malt-source quality", "Result": f"{summary['Q_malt']:.3f}"},
    {"Item": "Hop quality", "Result": f"{summary['Q_hop']:.3f}"},
    {"Item": "Yeast quality", "Result": f"{summary['Q_yeast']:.3f}"},
])
display(task35_solution_table)

def report_nonzero_ingredients(x, tol=1e-7):
    groups = [
        ("Barley", list(barley["name"]), x[slice_B()]),
        ("Malt", list(malts["name"]), x[slice_M()]),
        ("Malt extract", list(maltextracts["name"]), x[slice_E()]),
        ("Hops", list(hops["name"]), x[slice_H()]),
        ("Yeast", list(yeasts["name"]), x[slice_Y()]),
    ]
    rows = []
    for group, names, values in groups:
        for name, value in zip(names, values):
            if abs(value) > tol:
                rows.append({"Group": group, "Variable": name, "Amount (kg)": value})
    return pd.DataFrame(rows).round({"Amount (kg)": 3})

print("Table 8.3: Task 5 Non-Zero Ingredients")
task35_ingredients_table = report_nonzero_ingredients(best_solution["result"].x)
display(task35_ingredients_table)


Table 8.1: Task 5 Problem Setup


,Parameter,Value
0,Beer volume,5000 L
1,Colour,amber
2,EBC bounds,20 to 29
3,Malt-source quality,at least 3
4,Hop quality,at least 3
5,Yeast quality,at least 3


Table 8.2: Task 5 Optimal Solution Summary


,Item,Result
0,Total cost,4891.923
1,Process choice,"(0,1)"
2,Achieved EBC,20.000
3,Malt-source quality,3.000
4,Hop quality,3.000
5,Yeast quality,3.000


Table 8.3: Task 5 Non-Zero Ingredients


,Group,Variable,Amount (kg)
0,Malt,malt03,901.457
1,Malt,malt06,17.497
2,Malt,malt12,34.994
3,Hops,hop07,3.421
4,Hops,hop10,3.421
5,Yeast,yeast03,3.750
